# DeepSurv (PyCox) - Training Pipeline

Questo notebook definisce l'addestramento di un modello di Deep Learning per l'analisi di sopravvivenza (DeepSurv) utilizzando la libreria PyCox. A differenza dei modelli basati su alberi, le reti neurali richiedono una preparazione dei dati profondamente diversa e accorgimenti specifici per evitare l'overfitting.

---

### Fase 1: Caricamento e Preparazione Tensori
Il dataset viene caricato mantenendo la struttura di pulizia originale.
A differenza di RSF e XGBoost, in questo caso non imputiamo i valori "-1" o le mediane piatte con "NaN". La rete neurale gestirà questi valori numerici direttamente.
Inoltre, per compatibilità con PyTorch e PyCox:
* Tutte le feature (`X`) vengono convertite esplicitamente nel formato numerico `float32`.
* Il target (`Y`) non è un array strutturato di scikit-survival, ma viene trasformato in una tupla di tensori `(tempi, eventi)`.

### Fase 2: Split Multiplo (Train / Validation / Test) e Scaling
Il dataset subisce una doppia partizione:
1. **Test Set (25%):** Isolato per la valutazione finale.
2. **Validation Set (20% del Train residuo):** Creato appositamente per monitorare la rete durante l'addestramento e capire quando fermarsi.
* **Scaling Obbligatorio:** Le reti neurali sono estremamente sensibili alla scala delle variabili. Applichiamo uno `StandardScaler` (addestrato solo sul Train) per standardizzare tutte le feature. Lo scaler viene poi salvato (`.pkl`) perché sarà fondamentale per scalare correttamente i dati futuri o quelli del test set.

### Fase 3: Architettura della Rete Neurale (DeepSurv)
Costruiamo un Multi-Layer Perceptron (MLPVanilla) configurato per limitare la complessità ed evitare di memorizzare il dataset (overfitting):
* **Strati Nascosti:** Due layer densi molto piccoli (32 e 16 nodi) proporzionati alle dimensioni ridotte del dataset.
* **Batch Normalization:** Attivata per stabilizzare e accelerare la convergenza dei gradienti durante l'addestramento.
* **Dropout (0.4):** Una tecnica di regolarizzazione aggressiva che "spegne" casualmente il 40% dei neuroni ad ogni iterazione, costringendo la rete ad apprendere pattern robusti e distribuiti.

### Fase 4: Addestramento ed Early Stopping
L'addestramento utilizza l'ottimizzatore Adam con un learning rate di 0.01 per un massimo di 200 epoche.
Per garantire la massima generalizzazione, introduciamo una callback di **Early Stopping** (patience=15): se l'errore sul Validation Set smette di diminuire per 15 epoche consecutive, l'addestramento si interrompe automaticamente, prevenendo l'overfitting.

### Fase 5: Salvataggio Modello
A differenza dei modelli classici di scikit-learn, la rete neurale viene salvata nel formato nativo di PyTorch (`.pt`), mentre lo scaler viene salvato separatamente. Entrambi verranno ricaricati in fase di evaluation.

In [40]:
import os, warnings
import pandas as pd
import joblib
import torchtuples as tt
from pycox.models import CoxPH
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
def get_project_root(current_path):
    if os.path.exists(os.path.join(current_path, "data", "dataset_cleaned.csv")):
        return current_path
    parent = os.path.dirname(current_path)
    if parent == current_path:
        raise FileNotFoundError("Root del progetto non trovata.")
    return get_project_root(parent)

CWD = os.getcwd()
PROJECT_ROOT = get_project_root(CWD)

DATA_PATH   = os.path.join(PROJECT_ROOT, "data", "dataset_cleaned.csv")
MODEL_DIR   = os.path.join(PROJECT_ROOT, "models")
MODEL_PATH  = os.path.join(MODEL_DIR, "deepsurv_model.pt")
SCALER_PATH = os.path.join(MODEL_DIR, "deepsurv_scaler.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 20
TEST_SIZE    = 0.25
VAL_SIZE     = 0.20

# ==============================================================================
# FASE 1: CARICAMENTO E PREPROCESSING
# ==============================================================================
df_clean = pd.read_csv(DATA_PATH)

X = df_clean.drop(columns=[TIME_COL, EVENT_COL]).astype('float32')
get_target = lambda df: (df[TIME_COL].values.astype('float32'), df[EVENT_COL].values.astype('float32'))

# ==============================================================================
# FASE 2: SPLIT TRAIN / VAL / TEST E SCALING
# ==============================================================================
X_train_full, X_test, df_y_train_full, df_y_test = train_test_split(
    X, df_clean[[TIME_COL, EVENT_COL]],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_clean[EVENT_COL]
)

X_train, X_val, df_y_train, df_y_val = train_test_split(
    X_train_full, df_y_train_full,
    test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=df_y_train_full[EVENT_COL]
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype('float32')
X_val_scaled   = scaler.transform(X_val).astype('float32')

joblib.dump(scaler, SCALER_PATH)

y_train = get_target(df_y_train)
y_val   = get_target(df_y_val)

# ==============================================================================
# FASE 3: DEFINIZIONE ARCHITETTURA DEEPSURV
# ==============================================================================
in_features  = X_train_scaled.shape[1]
num_nodes    = [32, 16]
out_features = 1
batch_norm   = True
dropout      = 0.4

net = tt.practical.MLPVanilla(
    in_features, num_nodes, out_features,
    batch_norm, dropout
)

model = CoxPH(net, tt.optim.Adam(lr=0.01))

# ==============================================================================
# FASE 4: TRAINING E SALVATAGGIO
# ==============================================================================
batch_size = 32
epochs     = 200
callbacks  = [tt.callbacks.EarlyStopping(patience=15)]

log = model.fit(
    X_train_scaled, y_train,
    batch_size, epochs,
    callbacks,
    val_data=(X_val_scaled, y_val),
    verbose=False
)

model.save_net(MODEL_PATH)

# ==============================================================================
# RIEPILOGO FINALE
# ==============================================================================
print("=" * 62)
print("  TRAINING COMPLETATO - DEEPSURV (PyCox)")
print("=" * 62)
print(f"  Epoche effettuate : {len(log.to_pandas())}")
print(f"  Loss finale (Val) : {log.to_pandas()['val_loss'].iloc[-1]:.4f}")
print(f"  Artefatti salvati in: {MODEL_DIR}/")
print("=" * 62)

  TRAINING COMPLETATO - DEEPSURV (PyCox)
  Epoche effettuate : 37
  Loss finale (Val) : 3.3240
  Artefatti salvati in: C:\Users\nicon\PycharmProjects\ProgettoTesi\models/


# DeepSurv (PyCox) - Evaluation & Explainable AI

Questo notebook valuta le performance della rete neurale DeepSurv e ne esplora i meccanismi decisionali. Date le peculiarità architetturali di PyCox, la pipeline di valutazione e spiegabilità richiede alcuni passaggi di adattamento rispetto ai modelli standard.

---

### Fase 1: Ricostruzione Ambiente, Scaling e Baseline Hazards
Per valutare la rete, replichiamo rigorosamente l'ambiente di training:
* [cite_start]Ricarichiamo il dataset e ricreiamo il Test Set (25%) garantendo l'assenza di data leakage[cite: 47].
* [cite_start]Applichiamo lo `StandardScaler` salvato in addestramento per normalizzare il Test Set[cite: 47].
* [cite_start]Ricostruiamo la medesima architettura di rete (nodi 32-16, dropout 0.4) e carichiamo i pesi dal file `.pt`[cite: 48].
* **Baseline Hazards:** A differenza di altri modelli, DeepSurv in PyCox calcola un rischio relativo. [cite_start]Per ottenere funzioni di sopravvivenza assolute ($S(t)$) e calcolare l'Integrated Brier Score, dobbiamo prima calcolare le "Baseline Hazards" passando l'intero set di training alla rete[cite: 48].

### Fase 2: Metriche Globali di Sopravvivenza
Le probabilità predette vengono appiattite e passate alle metriche standard:
* [cite_start]**C-index:** Misura l'accuratezza del ranking di rischio predetto dalla rete[cite: 49].
* [cite_start]**Integrated Brier Score (IBS):** Il dataframe di sopravvivenza predetto da PyCox viene interpolato per allinearlo ai tempi temporali di valutazione richiesti da `sksurv`, ottenendo così una misura di calibrazione dell'errore nel tempo[cite: 49].
* [cite_start]**Time-dependent AUC:** Valuta l'AUC a step temporali specifici (1, 3 e 5 anni)[cite: 49].

### Fase 3: Stratificazione del Rischio (Validazione Clinica)
[cite_start]I pazienti del test set vengono separati in coorti ad "Alto Rischio" e "Basso Rischio" basandosi sulla mediana del rischio relativo (output grezzo della rete)[cite: 51].
[cite_start]Successivamente, si mediano le curve di sopravvivenza (derivate dal dataframe di PyCox) per verificare visivamente l'efficacia del modello nel differenziare le prognosi[cite: 51].

### Fase 4: Explainable AI Globale (Permutation Feature Importance)
Le reti neurali sono notoriamente delle "black box". [cite_start]Per ispezionarle globalmente, avvolgiamo il modello DeepSurv in una classe wrapper (`PyCoxWrapper`) compatibile con scikit-learn[cite: 52]. [cite_start]Questo permette di eseguire la Permutation Feature Importance, alterando una variabile scalata alla volta per misurare la conseguente diminuzione del C-index, rivelando così quali feature dominano l'apprendimento globale[cite: 52].

### Fase 5: Explainable AI Locale (SHAP) con Dati Raw
Per analizzare le decisioni sui singoli pazienti usiamo SHAP con un accorgimento tecnico cruciale:
[cite_start]Creiamo un wrapper personalizzato (`predict_risk_shap`) che accetta in input i dati **grezzi** (non scalati), applica lo scaler internamente e restituisce il rischio[cite: 54, 55].
[cite_start]Questo permette al KernelExplainer di SHAP di campionare e restituire grafici (Summary Plot e Waterfall Plot) che mostrano i **valori originali e comprensibili delle feature** (es. "Età = 65"), anziché valori astratti standardizzati (es. "Età = 1.24"), rendendo l'interpretazione clinicamente utile e intellegibile[cite: 55].

In [41]:
import os, warnings
import pandas as pd
import numpy as np
import joblib
import torchtuples as tt
import shap
import matplotlib.pyplot as plt
from pycox.models import CoxPH
from sksurv.util import Surv
from sksurv.metrics import (concordance_index_censored,
                            integrated_brier_score,
                            cumulative_dynamic_auc)
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
def get_project_root(current_path):
    if os.path.exists(os.path.join(current_path, "data", "dataset_cleaned.csv")):
        return current_path
    parent = os.path.dirname(current_path)
    if parent == current_path:
        raise FileNotFoundError("Root del progetto non trovata.")
    return get_project_root(parent)

CWD = os.getcwd()
PROJECT_ROOT = get_project_root(CWD)

if os.path.basename(CWD) == "DeepSurv":
    SCRIPT_DIR = CWD
else:
    SCRIPT_DIR = os.path.join(PROJECT_ROOT, "DeepSurv")

RESULTS_DIR = os.path.join(SCRIPT_DIR, "results")
DATA_PATH   = os.path.join(PROJECT_ROOT, "data", "dataset_cleaned.csv")
MODEL_DIR   = os.path.join(PROJECT_ROOT, "models")
MODEL_PATH  = os.path.join(MODEL_DIR, "deepsurv_model.pt")
SCALER_PATH = os.path.join(MODEL_DIR, "deepsurv_scaler.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

for _f, _name in [(MODEL_PATH, "deepsurv_model.pt"), (SCALER_PATH, "deepsurv_scaler.pkl")]:
    if not os.path.exists(_f):
        raise FileNotFoundError(f"{_name} non trovato in {MODEL_DIR}. Esegui prima il training.")

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 20
TEST_SIZE    = 0.25

# ==============================================================================
# 1. CARICAMENTO DATI, SCALER E MODELLO DEEPSURV
# ==============================================================================
print("=" * 62)
print("  EVALUATION & XAI - DEEPSURV (PyCox)")
print("=" * 62)

df_clean = pd.read_csv(DATA_PATH)

X = df_clean.drop(columns=[TIME_COL, EVENT_COL])
feature_cols = X.columns.tolist()

X_train_full, X_test, df_y_train_full, df_y_test = train_test_split(
    X, df_clean[[TIME_COL, EVENT_COL]],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_clean[EVENT_COL]
)

y_train_full_sksurv = Surv.from_arrays(event=df_y_train_full[EVENT_COL].astype(bool), time=df_y_train_full[TIME_COL])
y_test_sksurv = Surv.from_arrays(event=df_y_test[EVENT_COL].astype(bool), time=df_y_test[TIME_COL])

scaler = joblib.load(SCALER_PATH)
X_train_full_scaled = scaler.transform(X_train_full).astype('float32')
X_test_scaled = scaler.transform(X_test).astype('float32')

print(f"[+] Test set ricostruito: {len(X_test)} pazienti | {df_y_test[EVENT_COL].sum()} eventi")

in_features = X_train_full_scaled.shape[1]
num_nodes = [32, 16]
out_features = 1
batch_norm = True
dropout = 0.4

net = tt.practical.MLPVanilla(in_features, num_nodes, out_features, batch_norm, dropout)
model = CoxPH(net, tt.optim.Adam())
model.load_net(MODEL_PATH, weights_only=False)

y_train_full_pycox = (df_y_train_full[TIME_COL].values.astype('float32'), df_y_train_full[EVENT_COL].values.astype('float32'))
_ = model.compute_baseline_hazards(X_train_full_scaled, y_train_full_pycox)

risk_test = model.predict(X_test_scaled).flatten()
surv_df = model.predict_surv_df(X_test_scaled)

print("[+] Modello DeepSurv caricato correttamente\n")

# ==============================================================================
# FASE 1: METRICHE GLOBALI
# ==============================================================================
print("=" * 62)
print("  FASE 1: METRICHE GLOBALI")
print("=" * 62)

c_index = concordance_index_censored(y_test_sksurv['event'], y_test_sksurv['time'], risk_test)[0]

lb = np.percentile(y_test_sksurv['time'], 10)
ub = np.percentile(y_test_sksurv['time'], 90)
ibs_times = np.linspace(lb, ub, 20)

preds_interp = np.array([np.interp(ibs_times, surv_df.index.values, surv_df.iloc[:, i].values) for i in range(surv_df.shape[1])])
ibs = integrated_brier_score(y_train_full_sksurv, y_test_sksurv, preds_interp, ibs_times)

tempi_giorni = [365, 1095, 1825]
etichette    = ["1 anno (365 gg)", "3 anni (1095 gg)", "5 anni (1825 gg)"]
min_t, max_t = y_test_sksurv['time'].min(), y_test_sksurv['time'].max()

tempi_validi, label_validi = [], []
for t_g, lbl in zip(tempi_giorni, etichette):
    if min_t < t_g < max_t:
        tempi_validi.append(t_g)
        label_validi.append(lbl)

auc_vals, mean_auc = None, None
if tempi_validi:
    auc_vals, mean_auc = cumulative_dynamic_auc(y_train_full_sksurv, y_test_sksurv, risk_test, np.array(tempi_validi))

print(f"  C-index (test set) : {c_index:.4f}")
print(f"  IBS                : {ibs:.4f} (baseline ~0.25, ↓ meglio)")
if mean_auc:
    print(f"  Mean AUC           : {mean_auc:.4f}")
    for lbl, auc in zip(label_validi, auc_vals):
        print(f"    - AUC a {lbl} : {auc:.4f}")

# ==============================================================================
# FASE 2: STRATIFICAZIONE DEL RISCHIO
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 2: STRATIFICAZIONE CLINICA (Alto vs Basso Rischio)")
print("=" * 62)

median_risk = np.median(risk_test)
mask_high   = risk_test > median_risk
mask_low    = risk_test <= median_risk

mean_surv_high = surv_df.loc[:, mask_high].mean(axis=1)
mean_surv_low  = surv_df.loc[:, mask_low].mean(axis=1)
plot_times     = surv_df.index.values

fig1, ax1 = plt.subplots(figsize=(10, 6))
ax1.step(plot_times, mean_surv_low,  where="post", label="Basso Rischio", color="#2ca02c", lw=2.5)
ax1.step(plot_times, mean_surv_high, where="post", label="Alto Rischio", color="#d62728", lw=2.5)
ax1.fill_between(plot_times, mean_surv_low, mean_surv_high, alpha=0.08, color="grey", step="post")
ax1.set_title("Curve di Sopravvivenza (DeepSurv): Alto vs Basso Rischio", fontsize=13, fontweight="bold")
ax1.set_ylabel("Probabilità di Sopravvivenza", fontsize=11)
ax1.set_xlabel("Giorni di Follow-up", fontsize=11)
ax1.set_ylim(0, 1.02)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_survival_curves.png"), dpi=150)
plt.close()
print(f"  [+] Grafico curve salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 3: XAI GLOBALE (PERMUTATION FEATURE IMPORTANCE)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 3: PERMUTATION FEATURE IMPORTANCE")
print("=" * 62)
print("  Calcolo in corso (10 ripetizioni)...")

class PyCoxWrapper:
    def fit(self, X, y=None): return self
    def predict(self, X):
        return model.predict(X.astype("float32")).flatten()

def _score_fn(estimator, X, y):
    risk = estimator.predict(X)
    return concordance_index_censored(y["event"], y["time"], risk)[0]

perm = permutation_importance(
    PyCoxWrapper(), X_test_scaled, y_test_sksurv,
    scoring=_score_fn,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

imp_mean = np.maximum(perm.importances_mean, 0)
imp_pct  = 100.0 * imp_mean / (imp_mean.sum() + 1e-12)

df_imp = pd.DataFrame({
    "Feature":        feature_cols,
    "Importanza (%)": imp_pct,
    "Std":            perm.importances_std,
}).sort_values("Importanza (%)", ascending=False)

fig2, ax2 = plt.subplots(figsize=(9, max(6, len(feature_cols) * 0.35)))
ax2.barh(df_imp["Feature"][::-1], df_imp["Importanza (%)"][::-1], color="#1f77b4", xerr=df_imp["Std"][::-1], capsize=3)
ax2.set_xlabel("Importanza relativa (%)", fontsize=11)
ax2.set_title("Permutation Feature Importance – DeepSurv", fontsize=12, fontweight="bold")
ax2.grid(axis="x", alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_permutation_importance.png"), dpi=150)
plt.close()
print(f"  [+] Grafico Permutation salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 4: XAI LOCALE (SHAP)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 4: XAI LOCALE (SHAP)")
print("=" * 62)

def predict_risk_shap(X_raw):
    if isinstance(X_raw, pd.DataFrame):
        X_raw = X_raw.values
    X_scaled = scaler.transform(X_raw).astype('float32')
    return model.predict(X_scaled).flatten()

background = shap.sample(X_train_full, 100, random_state=RANDOM_STATE)
explainer_shap = shap.Explainer(predict_risk_shap, background)

print("  Calcolo SHAP su tutto il test set...")
shap_values_all = explainer_shap(X_test)

plt.figure(figsize=(10, max(6, len(feature_cols) * 0.35)))
shap.plots.beeswarm(shap_values_all, max_display=15, show=False)
plt.title("SHAP Summary – DeepSurv (Contributo al Risk Score)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_shap_summary.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  [+] SHAP Summary plot salvato")

idx_alto  = int(np.argmax(risk_test))
idx_basso = int(np.argmin(risk_test))
t_test_arr = df_y_test[TIME_COL].values
e_test_arr = df_y_test[EVENT_COL].values

for label, idx in [("ALTO RISCHIO", idx_alto), ("BASSO RISCHIO", idx_basso)]:
    plt.figure(figsize=(14, 7))
    try:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False)
    except TypeError:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False)

    ax_wf = plt.gca()
    plt.title(f"SHAP Waterfall – DeepSurv – Paziente {label}", fontsize=12, fontweight="bold", pad=12)
    ax_wf.tick_params(axis="y", labelsize=9)
    for text in ax_wf.texts:
        if "f(x)" not in text.get_text() and "base value" not in text.get_text():
            text.set_fontsize(8)
    plt.subplots_adjust(left=0.42, right=0.95, top=0.90, bottom=0.10)
    ax_wf.grid(axis="x", alpha=0.3, linestyle="--")

    fname = f"deepsurv_shap_waterfall_{label.lower().replace(' ', '_')}.png"
    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
    plt.close()

print("  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio")

# ==============================================================================
# RIEPILOGO FINALE
# ==============================================================================
print("\n" + "=" * 62)
print("  RIEPILOGO FINALE – DeepSurv (PyCox)")
print("=" * 62)
print(f"  ✅ Esecuzione completata. Tutti i grafici sono salvati in:\n  {RESULTS_DIR}")

  EVALUATION & XAI - DEEPSURV (PyCox)
[+] Test set ricostruito: 132 pazienti | 56 eventi
[+] Modello DeepSurv caricato correttamente

  FASE 1: METRICHE GLOBALI
  C-index (test set) : 0.7034
  IBS                : 0.2130 (baseline ~0.25, ↓ meglio)
  Mean AUC           : 0.7120
    - AUC a 1 anno (365 gg) : 0.7818
    - AUC a 3 anni (1095 gg) : 0.6667
    - AUC a 5 anni (1825 gg) : 0.7113

  FASE 2: STRATIFICAZIONE CLINICA (Alto vs Basso Rischio)
  [+] Grafico curve salvato in: C:\Users\nicon\PycharmProjects\ProgettoTesi\DeepSurv\results

  FASE 3: PERMUTATION FEATURE IMPORTANCE
  Calcolo in corso (10 ripetizioni)...
  [+] Grafico Permutation salvato in: C:\Users\nicon\PycharmProjects\ProgettoTesi\DeepSurv\results

  FASE 4: XAI LOCALE (SHAP)
  Calcolo SHAP su tutto il test set...
  [+] SHAP Summary plot salvato
  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio

  RIEPILOGO FINALE – DeepSurv (PyCox)
  ✅ Esecuzione completata. Tutti i grafici sono salvati in:
  C:\U